# Spark Bronze wikpedia page reads

In [1]:
exeuction_date = "2025-01-01"
full_refresh = True

In [2]:

import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def get_config(config_name):

    config_server_url = os.environ.get("TFDS_CONFIG_URL")
    if config_server_url is None:
        config_server_url = "http://tfds-config:8005/api/configs"

    config_url = config_server_url + "/" + config_name

    print(f"retrieving {config_name} config from {config_url}")
    response = requests.get(config_url)
    response.raise_for_status()
    if response.json() is None:
        raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
    cfg = response.json().get("config")
    if cfg is None:
        raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

    if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
        cfg["url"] = os.environ["TFDS_S3_URL"]
    if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
        cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
    return cfg


def get_spark_session():
    """Get spark client for s3."""
    s3_cfg = get_config("s3")
    spark_cfg = get_config("spark")

    print(f"using s3 endpoint: {s3_cfg['url']}")
    print(f"using spark master: {spark_cfg['master_url']}")

    spark_session = (  SparkSession
        .builder
        .master('spark://spark-master:7077')
        .appName("Wikipedia page reads - Bronze")
        .config("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
        .config("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
        .config("spark.hadoop.fs.s3a.endpoint", s3_cfg["url"])
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config(
            "spark.jars",
            "../jars/hadoop-aws-3.3.4.jar,../jars/aws-java-sdk-bundle-1.12.262.jar")
        .config("spark.executor.memory", "2g")
        .getOrCreate()
    )

    return spark_session


In [ ]:
from pyspark.sql.functions import input_file_name, col, sum as _sum, substring, to_date

spark = get_spark_session()

s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/12/pageviews-20250312-000000.gz"
s3_path = "s3a://data/pageviews-20250310-130000.gz"
s3_path = "s3a://data/wikipedia_pageviews/2025/2025-03/**/*.gz"

schema = StructType([
    StructField(name="domain_code", dataType=StringType(), nullable = True),
    StructField("page_title", StringType(), True),
    StructField("count_views", StringType(), True),
    # StructField("total_response_size", StringType(), True), this eems not to be populated, discard it
])
spark.sparkContext.setLogLevel("WARN")
print(f"Reading data from {s3_path}")

df_base = (
    spark.read.format("csv")
    .option("delimiter", " ")
    .option("header", "false")
    .option("inferSchema", "false")
    .schema(schema)
    .load(s3_path)
)

df_base = (
    df_base
    .na.drop(subset=["domain_code"])
    .filter(~col("page_title").contains(":"))
    .filter(~col("page_title").isin("-", 'Main_Page', 'Forside', 'Hauptseite', 'wiki.phtml'))
    .withColumn("country_code", substring(col("domain_code"), 1, 2))
    .filter(col("domain_code").isin("sv", 'dk', 'no', 'de', 'en'))
    .withColumn("count_views", col("count_views").cast(IntegerType()))
    .withColumn("file_name", input_file_name())
    .withColumn("date", to_date(substring(col("file_name"), -18, 8), 'yyyyMMdd'))
).repartition("date").cache()

# (   df_base
#     .write
#     .mode("overwrite")   # Options: 'overwrite', 'append', 'ignore', 'error' (default)
#     .format("parquet")    # Options: 'parquet', 'csv', 'json', 'orc', etc.
#     .saveAsTable("bronze.wikipedia_page_reads")
)

retrieving s3 config from http://tfds-config:8005/api/configs/s3
retrieving spark config from http://tfds-config:8005/api/configs/spark
using s3 endpoint: http://s3-minio:9000
using spark master: spark://spark-master:7077


25/04/08 15:34:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Reading data from s3a://data/wikipedia_pageviews/2025/2025-03/**/*.gz


25/04/08 15:34:37 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

aggregated_df = (
    df_base
    .groupBy('date', "page_title", "country_code")
    .agg(
        _sum(col("count_views")).alias("total_count_views"),
    )
)

national_win = (
    Window
    .partitionBy('date', "country_code")
    .orderBy(col("total_count_views").desc())
)

ranked_df = (
    aggregated_df
    .withColumn("national_rank", row_number().over(national_win))
    )

final_df = (
    ranked_df
    .filter(col('national_rank') <= 3)
    .orderBy('date', "national_rank", "country_code")
)

final_df.explain(mode="extended")

+----------+-----------------------------------+------------+-----------------+-------------+
|date      |page_title                         |country_code|total_count_views|national_rank|
+----------+-----------------------------------+------------+-----------------+-------------+
|2025-03-09|Thelma_&_Louise                    |de          |8505             |1            |
|2025-03-09|ICC_Champions_Trophy               |en          |42209            |1            |
|2025-03-09|Magnus_Brevig                      |no          |1325             |1            |
|2025-03-09|Jan_Stenbeck                       |sv          |2747             |1            |
|2025-03-09|Colonius                           |de          |6402             |2            |
|2025-03-09|Mark_Carney                        |en          |40059            |2            |
|2025-03-09|Jan-Erik_Aalbu                     |no          |914              |2            |
|2025-03-09|Mästarnas_mästare_2025             |sv          

In [ ]:
final_df.show(50, truncate=False)

In [7]:
ranked_df.explain(mode="extended")
# spark.stop()

== Parsed Logical Plan ==
'Project [date#26, page_title#1, country_code#10, total_count_views#70L, row_number() windowspecdefinition('date, 'country_code, 'total_count_views DESC NULLS LAST, unspecifiedframe$()) AS national_rank#76]
+- Aggregate [date#26, page_title#1, country_code#10], [date#26, page_title#1, country_code#10, sum(count_views#15) AS total_count_views#70L]
   +- RepartitionByExpression [date#26]
      +- Project [domain_code#0, page_title#1, count_views#15, country_code#10, file_name#20, to_date(substring(file_name#20, -18, 8), Some(yyyyMMdd), Some(Europe/Stockholm), false) AS date#26]
         +- Project [domain_code#0, page_title#1, count_views#15, country_code#10, input_file_name() AS file_name#20]
            +- Project [domain_code#0, page_title#1, cast(count_views#2 as int) AS count_views#15, country_code#10]
               +- Filter domain_code#0 IN (sv,dk,no,de,en)
                  +- Project [domain_code#0, page_title#1, count_views#2, substring(domain_code#0,

25/04/08 15:58:22 ERROR TaskSchedulerImpl: Lost executor 1 on 172.19.0.7: worker lost: Not receiving heartbeat for 60 seconds
25/04/08 15:58:22 ERROR TaskSchedulerImpl: Lost executor 0 on 172.19.0.6: worker lost: Not receiving heartbeat for 60 seconds
25/04/08 15:58:22 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_8_2 !
25/04/08 15:58:22 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_8_1 !
25/04/08 15:58:22 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_8_3 !
25/04/08 15:58:22 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_8_0 !
25/04/08 16:28:02 ERROR TaskSchedulerImpl: Lost executor 3 on 172.19.0.7: Command exited with code 0
25/04/08 16:28:03 ERROR TaskSchedulerImpl: Lost executor 2 on 172.19.0.6: Command exited with code 0
25/04/08 16:35:30 WARN TransportChannelHandler: Exception in connection from /192.168.100.160:65383
java.io.IOException: Operation timed out
	at java.base/sun.nio.ch.SocketDispat